In [1]:
from langchain_community.document_loaders import DataFrameLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

import pandas as pd

C:\Users\mahdi\AppData\Local\Temp\ipykernel_45244\3664084969.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DataFrameLoader


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
books = pd.read_csv("books_preprocessed.csv")
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


Load the descriptions from the dataframe

DONOTINCLUDE deviated from vid here: (can delete tagged_descriptions.txt)

In [4]:
# Load the descriptions directly from your existing DataFrame
loader = DataFrameLoader(books, page_content_column="tagged_description")
documents = loader.load()

documents[0]

Document(metadata={'isbn13': 9780002005883, 'isbn10': '0002005883', 'title': 'Gilead', 'authors': 'Marilynne Robinson', 'categories': 'Fiction', 'thumbnail': 'http://books.google.com/books/content?id=KQZCPgAACAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api', 'description': 'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best fri

In [5]:
# Initialize BAAI/bge-large-en-v1.5 locally
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cuda"}, # uses gpu for better processing
    encode_kwargs={"normalize_embeddings": True},
    query_encode_kwargs={
        "prompt": "Represent this sentence for searching relevant passages: ",
        "normalize_embeddings": True,
    }
)

# Build the vector database
db_books = Chroma.from_documents(
    documents,
    embedding=embeddings
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [6]:
sample_query = "heartwarming book involving dragons and wonder"
docs = db_books.similarity_search(sample_query, k = 10)

# prints just titles from the search results
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc.metadata.get('title')}")

# prints full details
docs

1. The Dragon's Eye
2. The Search for Power
3. The Dastard
4. Firedrake
5. The Last of the Really Great Whangdoodles
6. The Smoke Thief
7. Spindle's End
8. The Day of the Tempest
9. Wolfskin
10. Rose Daughter


[Document(id='d647074d-a468-4a1f-affd-254893a98092', metadata={'isbn13': 9780763628109, 'authors': 'Dugald Steer', 'categories': 'Juvenile Fiction', 'thumbnail': 'http://books.google.com/books/content?id=kasIop1plB4C&printsec=frontcover&img=1&zoom=1&source=gbs_api', 'ratings_count': 2139.0, 'isbn10': '0763628107', 'num_pages': 221.0, 'description': "When twelve-year-old Daniel Cook and his sister, Beatrice, spend the summer at a special school run by their parents' eccentric former tutor, they are introduced to the secret study of dragonology and find themselves caught up in an evil plot.", 'average_rating': 3.82, 'title_and_subtitle': "The Dragon's Eye", 'published_year': 2006.0, 'title': "The Dragon's Eye"}, page_content="9780763628109 When twelve-year-old Daniel Cook and his sister, Beatrice, spend the summer at a special school run by their parents' eccentric former tutor, they are introduced to the secret study of dragonology and find themselves caught up in an evil plot."),
 Docu

In [7]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3586,9780763628109,0763628107,The Dragon's Eye,Dugald Steer,Juvenile Fiction,http://books.google.com/books/content?id=kasIo...,When twelve-year-old Daniel Cook and his siste...,2006.0,3.82,221.0,2139.0,The Dragon's Eye,9780763628109 When twelve-year-old Daniel Cook...


In [8]:
def get_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k = 50)

    books_list = []

    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)].head(top_k)